In [ ]:
## critical for running the tutorial on jupyter notebook
## ignore if running on terminal
import nest_asyncio2  # type: ignore[import-untyped]

nest_asyncio2.apply()

In [3]:
import os
from dataclasses import dataclass
from enum import Enum, StrEnum, auto
from functools import total_ordering
from typing import Any, Literal, TypedDict, cast, get_type_hints
from pydantic import Field

from cognition import (
    Actuator,
    BinaryRelation,
    Cogent,
    DecisionProcess,
    DocEnum,
    Entity,
    EnumClassifier,
    IOContainer,
    Operator,
    Sensor,
    WorldGraph,
    operator_sorting_key,
    sorting_evaluator,
    FactDescriber,
    describe_facts
)

from utils import run_cogent

from pydantic import BaseModel
from pydantic_ai.models import infer_model

from smart_home.client import SmartHomeClient

In [4]:

assert "OPENAI_API_KEY" in os.environ, "Environment variable OPENAI_API_KEY is not set."
llm_model = infer_model("openai:gpt-4o")


In [5]:

# =========================================================================
# 1. Models & Relations
# =========================================================================

class DeviceState(StrEnum):
    """Smart home device operational states."""

    ON = "on"
    OFF = "off"
    LOCKED = "locked"
    UNLOCKED = "unlocked"
    IDLE = "idle"
    RUNNING = "running"
    EMPTY = "empty"
    FULL = "full"


class DeviceType(Enum):
    """Device capabilities/types."""

    LIGHT = "light"
    LOCK = "lock"
    RUNNABLE = "runnable"
    MOVABLE = "movable"
    FILLABLE = "fillable"

class Device(Entity):
    """Smart home device entity."""

    dtype: frozenset[DeviceType]
    dstate: DeviceState
    #name: str


DEVICE_TYPE_MAP: dict[str, frozenset[DeviceType]] = {
    "bedroom": frozenset({DeviceType.LIGHT}),
    "living_room": frozenset({DeviceType.LIGHT}),
    "bathroom": frozenset({DeviceType.LIGHT}),
    "kitchen": frozenset({DeviceType.LIGHT}),
    "dishwasher": frozenset({DeviceType.RUNNABLE}),
    "washer": frozenset({DeviceType.RUNNABLE}),
    "dryer": frozenset({DeviceType.RUNNABLE}),
    "laundry_basket": frozenset({DeviceType.MOVABLE, DeviceType.FILLABLE}),
    "main_door": frozenset({DeviceType.LOCK}),
}


class Room(Entity):
    """House room entity."""

    #name: str

class RoomName(StrEnum):
    KITCHEN = "room:kitchen"
    LIVING_ROOM = "room:living"
    BEDROOM = "room:bed"
    BATHROOM = "room:bath"
    LAUNDRY = "room:laundry"



class In(BinaryRelation):
    """Relationship indicating a device is in a room."""

    entity1: Device = Field(frozen=True)
    entity2: Room = Field(frozen=True)


# =========================================================================
# 3. Data Contracts & Enums
# =========================================================================


class Observation(BaseModel, frozen=True):
    """State observation payload across all environment devices."""

    bedroom: Literal[DeviceState.ON, DeviceState.OFF]
    living_room: Literal[DeviceState.ON, DeviceState.OFF]
    kitchen: Literal[DeviceState.ON, DeviceState.OFF]
    bathroom: Literal[DeviceState.ON, DeviceState.OFF]
    main_door: Literal[DeviceState.LOCKED, DeviceState.UNLOCKED]
    dishwasher: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    washer: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    dryer: Literal[DeviceState.IDLE, DeviceState.RUNNING]
    laundry_basket: Literal[DeviceState.EMPTY, DeviceState.FULL]


class ControlSignal(StrEnum):
    """Control signals for smart home devices."""

    TURN_ON = "turn_on"
    TURN_OFF = "turn_off"
    START = "start"
    STOP = "stop"
    FILL = "fill"
    EMPTY = "empty"
    LOCK = "lock"
    UNLOCK = "unlock"


@dataclass(frozen=True)
class Action:
    """Command action payload targeting a specific device."""

    device: str
    signal: ControlSignal


@dataclass(frozen=True)
class Utterance:
    """Natural language chat message contract."""

    phrase: str | None


class Mode(DocEnum):
    """Operating modes for the home assistant."""

    DAY = auto(), "day mode of smart assistant"
    EVENING = auto(), "evening mode of smart assistant"
    MIDNIGHT = auto(), "midnight mode of smart assistant"


class ResidentIntent(DocEnum):
    """Supported resident intent categories."""

    TURN_OFF = (
        "turn_off",
        "user is asking the assistant to turn a device off or stop a device",
    )
    TURN_ON = (
        "turn_on",
        "user is asking the assistant to turn a device on or start a device",
    )
    LOCK = "lock", "user is asking the assistant to lock a device"
    UNLOCK = "unlock", "user is asking the assistant to unlock a device"
    SET_MODE = "set_mode", "user is asking the assistant to set an operational mode"
    GET_DEVICE_INFO = "get_info", "user is asking the assistant to provide information about their devices"


class MyIntent(StrEnum):
    """Standard conversational responses from the assistant."""

    UNKNOWN = "I am sorry; I am not programmed to respond to that."
    CONFIRMATION = "Done."
    GREETING = "Hi!"
    GRATITUDE_ACK = "You are welcome."

In [6]:
class RoomsWorld(WorldGraph):
    """Graph auto-populated directly from strongly-typed dictionary specs."""

    def __init__(self) -> None:

        super().__init__()

        ## add bedroom to the world graph
        bedroom = Room(name=RoomName.BEDROOM)
        bedroom_lamp = Device(name="bedroom", dtype=DEVICE_TYPE_MAP["bedroom"], dstate=DeviceState.ON)
        self.add_entity(bedroom)
        self.add_entity(bedroom_lamp)
        self.add_relation(In(entity1=bedroom_lamp, entity2=bedroom))

        ## add living room to the world graph
        living_room = Room(name=RoomName.LIVING_ROOM)
        living_room_lamp = Device(name="living_room", dtype=DEVICE_TYPE_MAP["living_room"], dstate=DeviceState.ON)
        main_door = Device(name="main_door", dtype=DEVICE_TYPE_MAP["main_door"], dstate=DeviceState.LOCKED)
        self.add_entity(living_room)
        self.add_entity(living_room_lamp)
        self.add_entity(main_door)
        self.add_relation(In(entity1=living_room_lamp, entity2=living_room))
        self.add_relation(In(entity1=main_door, entity2=living_room))

        ## add bathroom to the world graph
        bathroom = Room(name=RoomName.BATHROOM)
        bathroom_lamp = Device(name="bathroom", dtype=DEVICE_TYPE_MAP["bathroom"], dstate=DeviceState.ON)
        self.add_entity(bathroom)
        self.add_entity(bathroom_lamp)
        self.add_relation(In(entity1=bathroom_lamp, entity2=bathroom))

        ## add kitchen to the world graph
        kitchen = Room(name=RoomName.KITCHEN)
        kitchen_lamp = Device(name="kitchen", dtype=DEVICE_TYPE_MAP["kitchen"], dstate=DeviceState.ON)
        dishwasher = Device(name="dishwasher", dtype=DEVICE_TYPE_MAP["dishwasher"], dstate=DeviceState.RUNNING)
        self.add_entity(kitchen)
        self.add_entity(kitchen_lamp)
        self.add_entity(dishwasher)
        self.add_relation(In(entity1=kitchen_lamp, entity2=kitchen))
        self.add_relation(In(entity1=dishwasher, entity2=kitchen))

        ## add laundry room to the world graph
        laundry = Room(name=RoomName.LAUNDRY)
        washer = Device(name="washer", dtype=DEVICE_TYPE_MAP["washer"], dstate=DeviceState.IDLE)
        dryer = Device(name="washer", dtype=DEVICE_TYPE_MAP["dryer"], dstate=DeviceState.IDLE)
        laundry_basket = Device(name="laundry_basket", dtype=DEVICE_TYPE_MAP["laundry_basket"], dstate=DeviceState.EMPTY)
        self.add_entity(washer)
        self.add_entity(dryer)
        self.add_entity(laundry)
        self.add_relation(In(entity1=washer, entity2=laundry))
        self.add_relation(In(entity1=dryer, entity2=laundry))


In [7]:
world = RoomsWorld()

In [8]:
world.snapshot

WorldSnapshot(items=frozenset({FrozenRoom(name='room:kitchen'), FrozenRoom(name='room:living'), FrozenIn(entity1=FrozenDevice(name='main_door', dtype=frozenset({<DeviceType.LOCK: 'lock'>}), dstate=<DeviceState.LOCKED: 'locked'>), entity2=FrozenRoom(name='room:living')), FrozenDevice(name='living_room', dtype=frozenset({<DeviceType.LIGHT: 'light'>}), dstate=<DeviceState.ON: 'on'>), FrozenIn(entity1=FrozenDevice(name='dishwasher', dtype=frozenset({<DeviceType.RUNNABLE: 'runnable'>}), dstate=<DeviceState.RUNNING: 'running'>), entity2=FrozenRoom(name='room:kitchen')), FrozenRoom(name='room:laundry'), FrozenDevice(name='bathroom', dtype=frozenset({<DeviceType.LIGHT: 'light'>}), dstate=<DeviceState.ON: 'on'>), FrozenIn(entity1=FrozenDevice(name='washer', dtype=frozenset({<DeviceType.RUNNABLE: 'runnable'>}), dstate=<DeviceState.IDLE: 'idle'>), entity2=FrozenRoom(name='room:laundry')), FrozenDevice(name='dishwasher', dtype=frozenset({<DeviceType.RUNNABLE: 'runnable'>}), dstate=<DeviceState.RUN

In [9]:
facts = wor

NameError: name 'wor' is not defined

In [24]:
facts = world.snapshot.by(In, lambda in1: isinstance(in1.entity1, Device) and in1.entity2.name == RoomName.KITCHEN)
#[print(fact) for fact in facts]

In [25]:
fact_list = [fact for fact in facts]
print(fact_list)

[FrozenIn(entity1=FrozenDevice(name='dishwasher', dtype=frozenset({<DeviceType.RUNNABLE: 'runnable'>}), dstate=<DeviceState.RUNNING: 'running'>), entity2=FrozenRoom(name='room:kitchen')), FrozenIn(entity1=FrozenDevice(name='kitchen', dtype=frozenset({<DeviceType.LIGHT: 'light'>}), dstate=<DeviceState.ON: 'on'>), entity2=FrozenRoom(name='room:kitchen'))]


In [26]:
describe_facts(instances=fact_list, task_desc="resident_asked: which devices in in my kitchen", llm=llm_model)

'In the kitchen, there is a running dishwasher and a light that is on.'

In [ ]:
describer = FactDescriber(In, "you are a smart home assistant responding to a resident")
describer(fact, [], llm_model, "resident_question: which device is in the kitchen?")

In [ ]:
class DeviceQuery(BaseModel):
    dtype: DeviceType | None
    dstate: DeviceState | None
    room: RoomName | None

In [ ]:
from cognition import ModelPopulator

filler = ModelPopulator(DeviceQuery, "help the resident understand the state of devices in their rooms")

In [ ]:
filler("which device is running in the kitchen", llm_model)

In [ ]:
filler("which device is in the kitchen", llm_model)

In [ ]:
filler("which device is in the rec room", llm_model)

In [ ]:
classifier = EnumClassifier(ResidentIntent, "What is the resident asking the assistant to do?")

In [ ]:
utterance = "what devices are in my kitchen"

In [ ]:
classifier(utterance, llm_model, num_trials=1)

In [ ]:
query = filler(utterance, llm_model)
query

In [ ]:
def formulate_device_query(in1) -> bool:
    is_device = isinstance(in1.entity1,Device)
    is_type = query.dtype in in1.entity1.dtype if query.dtype else True
    is_state = query.dstate is in1.entity1.dstate if query.dstate else True
    is_in_room = in1.entity2.name == query.room
    return is_device and is_type and is_state and is_in_room

In [ ]:
facts = world.snapshot.by(In, formulate_device_query)
[print(fact) for fact in facts]